# 9. Control Loop (Expanded)

Core logic that updates joint commands at each timestep using staged motion.

---

```python
def ControlLoop(self):
    self.time_ += self.dt

    t = self.time_
    d = self.stage_duration

    enable_value = 1.0
```

---

## 🧠 Big Picture: What This Function Is

This function is a **real-time controller**.

It runs:

```text
50 times per second (every 20 ms)
```

And each time it:

```text
1. Updates time
2. Determines current stage
3. Computes joint targets
4. Sends commands to the robot
```

---

## 🔁 The Control Loop Pattern

Every iteration follows this structure:

```text
loop:
    read state (already updated asynchronously)
    compute target
    send command
```

---

## ⏱️ Step 1: Advance Time

```python
self.time_ += self.dt
```

---

### What this does:

```text
Simulates a clock inside your controller
```

---

### Example:

| Iteration | time_ |
| --------- | ----- |
| 1         | 0.02  |
| 2         | 0.04  |
| 3         | 0.06  |

---

### 🧠 Key Insight

This is **not wall-clock time**—it is:

> A **discrete-time approximation of continuous motion**

---

## ⏳ Step 2: Define Time Variables

```python
t = self.time_
d = self.stage_duration
```

---

### Why define these?

* `t` → current time
* `d` → duration of each stage

---

### This enables:

```python
if t < d:
elif t < 2*d:
elif t < 3*d:
```

---

### Which creates:

```text
A time-based state machine
```

---

## 🎬 Time-Based State Machine

Your control loop implements:

```text
Time → determines behavior
```

---

### Visualization:

```text
0s      3s      6s      9s     15s     18s
|-------|-------|-------|------|-------|
 S1      S2      S3      S4     S5
```

---

### 🧠 Important Concept

This is a **deterministic controller**:

* Same inputs → same outputs
* No randomness
* Fully time-driven

---

## 🔌 Step 3: Enable Flag Initialization

```python
enable_value = 1.0
```

---

### What is this?

This controls whether the robot:

```text
1. Accepts SDK commands
2. Releases control back to internal controller
```

---

### Meaning:

| Value | Behavior         |
| ----- | ---------------- |
| 1.0   | SDK fully active |
| 0.0   | SDK disabled     |

---

### Where is it used?

Later:

```python
self.low_cmd.motor_cmd[G1JointIndex.kNotUsedJoint].q = enable_value
```

---

## ⚠️ Critical Insight

This is NOT a real joint:

```text
Index 29 = control flag
```

---

### So you are doing:

```text
"Send command AND set control mode"
```

---

## 🧠 Why Set It Here?

You initialize:

```text
enable_value = 1.0
```

so that:

* By default → robot follows your commands
* Later stages → may gradually disable control

---

## 🔄 Relationship to Stages

In later parts of the loop:

```python
enable_value = (1 - r)
```

---

### This creates:

```text
Gradual handover:
SDK → Robot internal controller
```

---

## ⚙️ System-Level Interpretation

At this point in the loop, you have:

| Variable       | Meaning           |
| -------------- | ----------------- |
| `time_`        | internal clock    |
| `t`            | current time      |
| `d`            | stage duration    |
| `enable_value` | control authority |

---

## 🔬 Control Theory Perspective

This function is implementing:

> A **discrete-time controller**

---

### Formally:

[
x_{k+1} = f(x_k, u_k)
]

Where:

* ( x_k ) = state (from `LowState`)
* ( u_k ) = command (your `LowCmd`)
* ( k ) = timestep

---

## 🤖 RL Interpretation

This is your **environment loop**:

```text
state  → already available
action → computed here
```

---

### Equivalent RL pseudocode:

```python
while True:
    state = observe()
    action = policy(state)
    apply(action)
```

---

## ⚠️ Important Subtlety

Notice:

```python
self.time_ += self.dt
```

instead of:

```python
time.time()
```

---

### Why?

* Ensures deterministic behavior
* Avoids jitter from system clock
* Keeps timing tied to control loop

---

## 🔄 What Happens Next?

After this setup, the loop will:

```text
1. Select a stage (based on time)
2. Compute joint targets (using interpolation)
3. Apply PD control parameters
4. Send command to robot
```

---

## 🔥 Why This Section Matters So Much

This is where:

* Timing becomes motion
* State becomes action
* Code becomes physical behavior

---

## 🧠 Teaching Insight

This is the perfect place to emphasize:

> “Robots are controlled by continuously updating commands, not one-time instructions.”

---

Students should understand:

* Motion = repeated updates
* Control = continuous process
* Timing = everything

---

## 🔄 Analogy

Think of this like:

```text
A musician playing notes in time
```

* Each loop iteration = one note
* Timing determines rhythm
* Sequence determines behavior

---

## 🚀 Summary

This section:

| Step                   | Role                            |
| ---------------------- | ------------------------------- |
| Update time            | Drives motion progression       |
| Define `t`, `d`        | Enables stage logic             |
| Initialize enable flag | Controls authority              |
| Sets loop context      | Prepares for motion computation |

---

> 🔥 This is the **engine of your robot controller**—everything else plugs into this loop.

